# CGM Preprocessing

In [ ]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive

    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
else:
    PROJECT_ROOT = Path('..').resolve()

DATA_ROOT = PROJECT_ROOT
print(f"DATA_ROOT: {DATA_ROOT}")

Mounted at /content/drive
DATA_ROOT: /content/drive/MyDrive/dissertation


In [ ]:
import os
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

In [ ]:
# -- Pipeline configuration --
NATIVE_STEP_MIN   = 5
WINDOW_MINUTES    = 120
WINDOW_STEPS      = WINDOW_MINUTES // NATIVE_STEP_MIN
HORIZON_MINUTES   = [15, 30, 60, 90, 120]
HORIZON_STEPS     = [h // NATIVE_STEP_MIN for h in HORIZON_MINUTES]  # = [3, 6, 12, 18, 24]
STEP_STEPS        = 1
MAX_GAP_STEPS      = 3
IMAGE_LOOKBACK_MIN = 120

EXCLUDE_PIDS = {'CGMacros-007', 'CGMacros-018', 'CGMacros-026'}

OUT_DIR = os.path.join(DATA_ROOT,'processed_data','processed_5min')
os.makedirs(OUT_DIR, exist_ok=True)

# Fail early if the PATH is wrong
CGM_ROOT = DATA_ROOT / 'CGMacros'
if not CGM_ROOT.exists():
    raise FileNotFoundError(
        f'Could not find {CGM_ROOT}. Edit PROJECT_ROOT above so it points to the folder '
        'that directly contains the CGMacros directory.'
    )


print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'CGMacros data: {CGM_ROOT}')
print(f"WINDOW_STEPS={WINDOW_STEPS}  HORIZON_STEPS={HORIZON_STEPS}  "
      f"MAX_GAP_STEPS={MAX_GAP_STEPS}  OUT_DIR={OUT_DIR}")

PROJECT_ROOT: /content/drive/MyDrive/dissertation
CGMacros data: /content/drive/MyDrive/dissertation/CGMacros
WINDOW_STEPS=24  HORIZON_STEPS=[3, 6, 12, 18, 24]  MAX_GAP_STEPS=3  OUT_DIR=/content/drive/MyDrive/dissertation/processed_native5min


In [ ]:
def load_full_grid(pid):
    df = pd.read_csv(os.path.join(DATA_ROOT, 'CGMacros', pid, f'{pid}.csv'))
    df['Timestamp'] = pd.to_datetime(df['Timestamp'])
    df = df.set_index('Timestamp').sort_index()
    gl = df['Dexcom GL']
    valid = gl.dropna()
    full_idx = pd.date_range(valid.index.min(), valid.index.max(), freq='1min')
    return gl.reindex(full_idx)


def find_breakpoints(gl_1min):
    d2 = gl_1min.diff().diff()
    return d2.abs() > 1e-6


def find_real_readings(gl_1min):
    return find_breakpoints(gl_1min).shift(-1, fill_value=False)


def dominant_phase(gl_1min, min_real=10):
    is_real = find_real_readings(gl_1min)
    if is_real.sum() < min_real:
        return None, 0.0
    counts = (gl_1min.index[is_real].minute % 5).value_counts(normalize=True)
    return int(counts.index[0]), float(counts.iloc[0])

In [ ]:
# extract the 5-minute series from 1-minute series
def extract_native_5min(gl_1min, phase):
    native_index = gl_1min.index[gl_1min.index.minute % 5 == phase]
    native_index = pd.date_range(native_index.min(), native_index.max(),
                                  freq=f'{NATIVE_STEP_MIN}min')
    return gl_1min.reindex(native_index)

# Demonstrate on CGMacros-001
pid_demo = 'CGMacros-001'
gl1_demo = load_full_grid(pid_demo)
phase_demo, purity_demo = dominant_phase(gl1_demo)
native_demo = extract_native_5min(gl1_demo, phase_demo)

print(f"{pid_demo}: phase={phase_demo}  purity={purity_demo:.1%}")
print(f"1-minute grid: {len(gl1_demo)} rows, {gl1_demo.notna().sum()} non-null")
print(f"native 5-minute grid: {len(native_demo)} rows, {native_demo.notna().sum()} non-null "
      f"({native_demo.isna().sum()} genuine gaps)")

CGMacros-001: phase=4  purity=100.0%
1-minute grid: 14275 rows, 14265 non-null
native 5-minute grid: 2855 rows, 2853 non-null (2 genuine gaps)


In [ ]:
# Gap-limited Linear Interpolation on Native Grid
def interpolate_native(native_series, max_gap_steps=MAX_GAP_STEPS):
    return native_series.interpolate(method='linear', limit=max_gap_steps,
                                      limit_direction='forward')

# Demonstrate again on CGMacros-001
native_demo_interp = interpolate_native(native_demo)
before = native_demo.isna().sum()
after = native_demo_interp.isna().sum()
print(f"{pid_demo}: {before} missing native points before gap-limited interpolation, "
      f"{after} after (gap limit = {MAX_GAP_STEPS} native steps = "
      f"{MAX_GAP_STEPS * NATIVE_STEP_MIN} minutes)")

CGMacros-001: 2 missing native points before gap-limited interpolation, 0 after (gap limit = 3 native steps = 15 minutes)


## 5. Meal-Image Matching

In [ ]:
# Meal Image Matching
def get_meal_info(df_raw, t_end, participant_dir, lookback_min=IMAGE_LOOKBACK_MIN):
    mask = (df_raw.index <= t_end) & df_raw['Meal Type'].notna()
    meal_rows = df_raw[mask]
    if meal_rows.empty:
        return None, None, False  # no meal logged yet at all for this participant

    last_meal = meal_rows.iloc[-1]
    meal_time = meal_rows.index[-1]
    time_since_meal = float((t_end - meal_time).total_seconds() / 60.0)

    has_meal = time_since_meal <= lookback_min
    if not has_meal:
        return None, time_since_meal, False  # meal too long ago; time gap is still real

    if pd.isna(last_meal['Image path']):
        return None, time_since_meal, True  # meal happened, was never photographed at all

    abs_path = os.path.join(participant_dir, last_meal['Image path'])
    if not os.path.exists(abs_path):
        return None, time_since_meal, True  # meal happened, photo missing/corrupted on disk

    return abs_path, time_since_meal, True

## 6. Sliding-Window Sample Construction

In [ ]:
## Sliding-Window Sample Construction
def create_samples(native_series, df_raw, pid, participant_dir):
    samples = []
    max_horizon_steps = max(HORIZON_STEPS)
    n = len(native_series)

    for i in range(0, n - WINDOW_STEPS - max_horizon_steps, STEP_STEPS):
        cgm_window = native_series.iloc[i: i + WINDOW_STEPS].values
        if np.isnan(cgm_window).any():
            continue

        t_end = native_series.index[i + WINDOW_STEPS - 1]

        targets = {}
        valid = True
        for h_min, h_steps in zip(HORIZON_MINUTES, HORIZON_STEPS):
            target_idx = i + WINDOW_STEPS - 1 + h_steps
            if target_idx >= n:
                valid = False
                break
            target_val = native_series.iloc[target_idx]
            if np.isnan(target_val):
                valid = False
                break
            targets[h_min] = float(target_val)
        if not valid:
            continue

        image_path, time_since_meal, has_meal = get_meal_info(df_raw, t_end, participant_dir)

        samples.append({
            'participant_id':  pid,
            'timestamp':       t_end,
            'cgm_sequence':    cgm_window.astype(np.float32),
            'image_path':      image_path,
            'has_image':       image_path is not None,
            'has_meal':        has_meal,
            'time_since_meal': time_since_meal,
            **{f'target_{h}': targets[h] for h in HORIZON_MINUTES},
        })
    return samples

In [ ]:
## Run the Full Pipeline Across All Participants
cgm_root = os.path.join(DATA_ROOT, 'CGMacros')
all_pids = sorted(d for d in os.listdir(cgm_root) if d.startswith('CGMacros-'))
kept_pids = [p for p in all_pids if p not in EXCLUDE_PIDS]

print(f"Total participants: {len(all_pids)}  Excluded: {sorted(EXCLUDE_PIDS)}  "
      f"Kept: {len(kept_pids)}")

all_samples = []
phase_report = []
native_by_pid = {}

for pid in kept_pids:
    pdir = os.path.join(cgm_root, pid)
    gl1 = load_full_grid(pid)
    phase, purity = dominant_phase(gl1)
    phase_report.append({'participant_id': pid, 'phase': phase, 'purity': purity})
    if phase is None or purity < 0.95:
        print(f"  !! {pid}: phase detection failed or unstable (purity={purity:.2f}) -- skipping")
        continue

    native_raw = extract_native_5min(gl1, phase)
    native = interpolate_native(native_raw)
    native_by_pid[pid] = (native_raw, native)

    df_raw = pd.read_csv(os.path.join(pdir, f'{pid}.csv'))
    df_raw['Timestamp'] = pd.to_datetime(df_raw['Timestamp'])
    df_raw = df_raw.set_index('Timestamp').sort_index()

    all_samples.extend(create_samples(native, df_raw, pid, pdir))

phase_report = pd.DataFrame(phase_report)
print(f"\nAll {len(kept_pids)} kept participants: phase purity range "
      f"[{phase_report['purity'].min():.3f}, {phase_report['purity'].max():.3f}]")
print(f"Total samples: {len(all_samples):,}")
n_img = sum(s['has_image'] for s in all_samples)
print(f"Samples with image: {n_img:,} ({100*n_img/len(all_samples):.1f}%)")

Total participants: 45  Excluded: ['CGMacros-007', 'CGMacros-018', 'CGMacros-026']  Kept: 42

All 42 kept participants: phase purity range [1.000, 1.000]
Total samples: 117,113
Samples with image: 33,188 (28.3%)


## Participant-Level Stratified Split (30 / 6 / 6, Balanced by Glycaemic Group)

In [ ]:
bio = pd.read_csv(os.path.join(CGM_ROOT, 'bio.csv'))
bio['participant_id'] = bio['subject'].apply(lambda s: f'CGMacros-{s:03d}')

def glycemic_group(a1c):
    if a1c < 5.7:
        return 'Healthy'
    elif a1c < 6.5:
        return 'Prediabetes'
    return 'Type2Diabetes'

bio['group'] = bio['A1c PDL (Lab)'].apply(glycemic_group)
bio = bio[bio['participant_id'].isin(kept_pids)].set_index('participant_id')

print(bio['group'].value_counts())
assert set(bio['group'].value_counts()) == {14}, "expected exactly 14 per group after exclusions"

pid_mean_gl = {}
for s in all_samples:
    pid_mean_gl.setdefault(s['participant_id'], []).append(s['cgm_sequence'].mean())
pid_mean_gl = {p: np.mean(v) for p, v in pid_mean_gl.items()}

train_pids, val_pids, test_pids = set(), set(), set()
for grp in ['Healthy', 'Prediabetes', 'Type2Diabetes']:
    grp_pids = sorted(bio.index[bio['group'] == grp], key=lambda p: pid_mean_gl[p])
    val_pids.update([grp_pids[4], grp_pids[9]])
    test_pids.update([grp_pids[1], grp_pids[12]])
    train_pids.update(p for p in grp_pids if p not in val_pids and p not in test_pids)

splits = {'train': [], 'val': [], 'test': []}
for s in all_samples:
    if s['participant_id'] in train_pids:
        splits['train'].append(s)
    elif s['participant_id'] in val_pids:
        splits['val'].append(s)
    elif s['participant_id'] in test_pids:
        splits['test'].append(s)

for name, pids_ in [('train', train_pids), ('val', val_pids), ('test', test_pids)]:
    grp_counts = bio.loc[list(pids_), 'group'].value_counts().to_dict()
    print(f"{name:5s}: {len(pids_)} participants {grp_counts}, {len(splits[name]):,} samples")

group
Healthy          14
Type2Diabetes    14
Prediabetes      14
Name: count, dtype: int64
train: 30 participants {'Healthy': 10, 'Type2Diabetes': 10, 'Prediabetes': 10}, 83,532 samples
val  : 6 participants {'Type2Diabetes': 2, 'Prediabetes': 2, 'Healthy': 2}, 16,739 samples
test : 6 participants {'Healthy': 2, 'Type2Diabetes': 2, 'Prediabetes': 2}, 16,842 samples


In [ ]:
# Normalization Statistics

train_samples = splits['train']
seqs = np.concatenate([s['cgm_sequence'] for s in train_samples])
targets = np.array([[s[f'target_{h}'] for h in HORIZON_MINUTES] for s in train_samples]).ravel()
all_vals = np.concatenate([seqs, targets])

cgm_stats = {'cgm_mean': float(all_vals.mean()), 'cgm_std': float(all_vals.std())}
print(f"cgm_mean = {cgm_stats['cgm_mean']:.2f} mg/dL, cgm_std = {cgm_stats['cgm_std']:.2f} mg/dL")

cgm_mean = 140.10 mg/dL, cgm_std = 42.13 mg/dL


In [ ]:
## Save Outputs
for name, split_samples in splits.items():
    with open(os.path.join(OUT_DIR, f'{name}.pkl'), 'wb') as f:
        pickle.dump(split_samples, f)

with open(os.path.join(OUT_DIR, 'cgm_stats.json'), 'w') as f:
    json.dump(cgm_stats, f, indent=2)

with open(os.path.join(OUT_DIR, 'split_participants.json'), 'w') as f:
    json.dump({'train': sorted(train_pids), 'val': sorted(val_pids),
               'test': sorted(test_pids), 'excluded': sorted(EXCLUDE_PIDS)}, f, indent=2)

print(f"Saved train/val/test .pkl + cgm_stats.json + split_participants.json -> {OUT_DIR}")

Saved train/val/test .pkl + cgm_stats.json + split_participants.json -> /content/drive/MyDrive/dissertation/processed_native5min


In [ ]:
gap_events = []
for pid, (native_raw, _) in native_by_pid.items():
    is_na = native_raw.isna().values
    i, n = 0, len(is_na)
    while i < n:
        if is_na[i]:
            j = i
            while j < n and is_na[j]:
                j += 1
            gap_events.append({'participant_id': pid, 'gap_steps': j - i, 'gap_minutes': (j - i) * 5})
            i = j
        else:
            i += 1

gaps = pd.DataFrame(gap_events)
print(f"{len(gaps)} independent gap events across {len(native_by_pid)} participants")
print(f"mean {gaps.gap_minutes.mean():.1f} min, median {gaps.gap_minutes.median():.1f} min")
print(gaps.gap_minutes.value_counts().sort_index())

fillable = gaps[gaps.gap_steps <= MAX_GAP_STEPS]
toolong = gaps[gaps.gap_steps > MAX_GAP_STEPS]
print(f"\n<= {MAX_GAP_STEPS * NATIVE_STEP_MIN} min (fully filled): {len(fillable)} "
      f"({100*len(fillable)/len(gaps):.1f}%)")
print(f"> {MAX_GAP_STEPS * NATIVE_STEP_MIN} min (left missing): {len(toolong)} "
      f"({100*len(toolong)/len(gaps):.1f}%), mean {toolong.gap_minutes.mean():.1f} min")
print("-> gap lengths are sharply bimodal with nothing observed between the two limits above; "
      "raising the limit into that empty region would recover nothing.")

59 independent gap events across 42 participants
mean 10.0 min, median 5.0 min
gap_minutes
5     47
10     4
15     2
30     1
35     1
40     1
50     1
65     2
Name: count, dtype: int64

<= 15 min (fully filled): 53 (89.8%)
> 15 min (left missing): 6 (10.2%), mean 47.5 min
-> gap lengths are sharply bimodal with nothing observed between the two limits above; raising the limit into that empty region would recover nothing.


In [ ]:
tot_lost_img = tot_kept_img = tot_lost_noimg = tot_kept_noimg = 0
lost_image_records = []

for pid in kept_pids:
    native_raw, native = native_by_pid[pid]
    vals, times = native.values, native.index.values
    n = len(vals)
    max_h = max(HORIZON_STEPS)

    pdir = os.path.join(cgm_root, pid)
    df_raw = pd.read_csv(os.path.join(pdir, f'{pid}.csv'))
    df_raw['Timestamp'] = pd.to_datetime(df_raw['Timestamp'])
    df_raw = df_raw.set_index('Timestamp').sort_index()
    img_notna = df_raw['Image path'].notna()

    def has_image_near(t_end):
        lookback_start = t_end - pd.Timedelta(minutes=IMAGE_LOOKBACK_MIN)
        return ((df_raw.index >= lookback_start) & (df_raw.index <= t_end) & img_notna).any()

    for i in range(0, n - WINDOW_STEPS - max_h):
        window = vals[i:i + WINDOW_STEPS]
        bad = np.isnan(window).any()
        if not bad:
            for h in HORIZON_STEPS:
                if np.isnan(vals[i + WINDOW_STEPS - 1 + h]):
                    bad = True
                    break
        t_end = pd.Timestamp(times[i + WINDOW_STEPS - 1])
        img = has_image_near(t_end)
        if img:
            if bad:
                tot_lost_img += 1
            else:
                tot_kept_img += 1
        else:
            if bad:
                tot_lost_noimg += 1
            else:
                tot_kept_noimg += 1

total_img, total_noimg = tot_lost_img + tot_kept_img, tot_lost_noimg + tot_kept_noimg
print(f"Image-bearing candidates: {total_img}, discarded {tot_lost_img} "
      f"({100*tot_lost_img/total_img:.3f}%)")
print(f"Non-image candidates: {total_noimg}, discarded {tot_lost_noimg} "
      f"({100*tot_lost_noimg/total_noimg:.3f}%)")
print(f"Relative risk: {(tot_lost_img/total_img)/(tot_lost_noimg/total_noimg):.2f}x")
print("-> small in absolute terms, but a real, documented skew -- plausibly because reopening the "
      "study app to photograph a meal is the same interaction that resumes a lapsed CGM sync.")

Image-bearing candidates: 39298, discarded 157 (0.400%)
Non-image candidates: 78101, discarded 129 (0.165%)
Relative risk: 2.42x
-> small in absolute terms, but a real, documented skew -- plausibly because reopening the study app to photograph a meal is the same interaction that resumes a lapsed CGM sync.
